In [ ]:
# python standard library imports
from typing import Self, Any
from pathlib import Path

In [ ]:
# model building imports
from keras import Model, Sequential, Input, layers

In [ ]:
# model training imports
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy
from keras.metrics import CategoricalAccuracy, AUC, F1Score
from keras.callbacks import ModelCheckpoint, CSVLogger, LearningRateScheduler

In [ ]:
# other imports
from keras.utils import image_dataset_from_directory

In [ ]:
class MyCNN(Model):
    """
    MyCNN class, inherets from keras' Model class
    """
    def __init__(self: Self, augmentation_layer, conv_configs, dense_configs, num_classes, activation: str = "relu"):
        """
        Initialization
        """

        super().__init__(name="my_cnn")
        
        self.augmentation_layer = augmentation_layer

        self.blocks = []
        for id, (filters, kernel, stride) in enumerate(conv_configs):
            block = {
                'conv': layers.Conv2D(filters, kernel, strides=stride, name=f"conv_layer_{id}", padding='same'),
                'bn': layers.BatchNormalization(),
                'actv': layers.Activation(activation),
                # Projeção 1x1 para alinhar canais/dimensões se necessário
                'shortcut': layers.Conv2D(filters, (1, 1), strides=stride, padding='same')
            }
            self.blocks.append(block)

        # Global Average Pooling em vez de Flatten
        self.gap = layers.GlobalAveragePooling2D(name="GAP_layer")
        
        self.dense_layers = [layers.Dense(u, name=f"dense_layer_{id}", activation=activation) for id, u in enumerate(dense_configs)]
        self.classifier = layers.Dense(num_classes, name="classification_head", activation='softmax')

    def call(self, inputs, training=False):
        """
        Forward call
        """

        x = inputs
        
        for b in self.blocks:
            shortcut = b['shortcut'](x) # Ajusta a identidade para a nova forma
            
            x = b['conv'](x)
            x = b['bn'](x, training=training)
            x = b['actv'](x)
            
            # Agora as formas (shapes) são compatíveis para a soma
            x = layers.Add()([x, shortcut])
            x = b['actv'](x) # Ativação após a soma (padrão ResNet)

        x = self.gap(x)
        for layer in self.dense_layers:
            x = layer(x)
            
        return self.classifier(x)

In [ ]:

# Configurações básicas
img_size = (224, 224) # Ajusta para o que o teu modelo espera
epochs = 64
batch_size = 32
data_dir_path = "wikiart_split"

# Carregar Treino
train_ds = image_dataset_from_directory(
    data_dir_path / 'train',
    image_size=img_size,
    batch_size=batch_size,
    label_mode='categorical',
    interpolation="bilinear"
)

# Carregar Validação
val_ds = image_dataset_from_directory(
    data_dir_path / 'val',
    image_size=img_size,
    batch_size=batch_size,
    label_mode='categorical',
    interpolation="bilinear"
)

# Carregar Teste
test_ds = image_dataset_from_directory(
    data_dir_path / 'test',
    image_size=img_size,
    batch_size=batch_size,
    label_mode='categorical',
    interpolation="bilinear",
    shuffle=False # Importante para avaliação final estável
)


In [ ]:
#Params
augmentation_layer = layers.Pipeline(
    [
        layers.RandomBrightness(factor=0.1, value_range=(0.0, 1.0)),
        layers.RandomFlip(),
        layers.RandomRotation(factor=0.1, fill_mode="reflect")
    ],
    name="augmentation_layer"
)
# Formato: (filtros, kernel, stride)
# Usamos strides=2 para reduzir a imagem em vez de MaxPool (mais moderno)
conv_setup = [
    (64, (7,7), 2),   # Camada inicial "Stem" (captura specs gerais)
    (64, (3,3), 1),   # Bloco Residual 1
    (128, (3,3), 2),  # Bloco Residual 2 (reduz dimensão)
    (128, (3,3), 1),  # Bloco Residual 3
    (256, (3,3), 2),  # Bloco Residual 4 (reduz dimensão)
    (256, (3,3), 1),  # Bloco Residual 5
    (512, (3,3), 2),  # Bloco Residual 6 (alta abstração)
    (512, (3,3), 1)   # Bloco Residual 7
]

# Camadas densas após o GAP
dense_setup = [1024, 512]
n_classes = 23

In [ ]:
# add L2 weight decay to the optimizer directly, don't add a new loss term
model = MyCNN(augmentation_layer=augmentation_layer, conv_configs=conv_setup, dense_configs=dense_setup, num_classes=n_classes)
optimizer = SGD(learning_rate=0.01, name="optimizer", weight_decay=0.01)
loss = CategoricalCrossentropy(name="loss")

In [ ]:
# metrics
categorical_accuracy = CategoricalAccuracy(name="accuracy")
auc = AUC(name="auc")
f1_score = F1Score(average="macro", name="f1_score")
metrics = [categorical_accuracy, auc, f1_score]

In [ ]:
# traces the computation
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

In [ ]:
# What are callbacks?
root_dir_path = Path(".")
checkpoint_file_path = root_dir_path / "checkpoint.keras"
metrics_file_path = root_dir_path = root_dir_path / "metrics.csv"

checkpoint_callback = ModelCheckpoint(
    checkpoint_file_path,
    monitor="val_loss",
    verbose=0
)
metrics_callback = CSVLogger(metrics_file_path)

In [ ]:
# What is a learning rate scheduler ?
def exp_decay_lr_scheduler(
    epoch: int,
    current_lr: float,
    factor: float = 0.975,
) -> float:
    """
    Exponential decay learning rate scheduler
    """

    current_lr *= factor

    return current_lr

In [ ]:
lr_scheduler_callback = LearningRateScheduler(exp_decay_lr_scheduler)

In [ ]:
callbacks = [
    checkpoint_callback,
    metrics_callback,
    lr_scheduler_callback
]

In [ ]:
# train the model
_ = model.fit(
    train_ds,
    validation_data=val_ds,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    verbose=2
)

In [ ]:
# evaluate on the test set
model.evaluate(
    test_ds,
    batch_size=batch_size,
    return_dict=True,
    verbose=0
)